In [6]:
import pandas as pd
import numpy as np

# ── Load dataset ─────────────────────────────────────────────────
print("Loading final dataset...")
df = pd.read_excel(r"1_Record.xlsx")
df['charttime'] = pd.to_datetime(df['charttime'])

print(f"  Loaded {len(df):,} rows, {df['subject_id'].nunique():,} patients")

# ── Pick first 100 unique patients ───────────────────────────────
first_100 = df['subject_id'].unique()[:100]
sample = df[df['subject_id'].isin(first_100)].copy()
print(f"  Running logic on {sample['subject_id'].nunique()} patients "
      f"({len(sample):,} rows)")

# ═══════════════════════════════════════════════════════════════
# HELPER — safe column read (returns NaN if column missing/null)
# ═══════════════════════════════════════════════════════════════
def get(row, col, default=np.nan):
    val = row.get(col, default)
    if pd.isna(val):
        return default
    return val

# ═══════════════════════════════════════════════════════════════
# 7-STEP LOGIC ENGINE
# ═══════════════════════════════════════════════════════════════
def run_logic(row):
    result = {
        'subject_id':      row['subject_id'],
        'hadm_id':         row['hadm_id'],
        'charttime':       row['charttime'],
        'step1_emergency': 'SAFE',
        'step2_fluid':     'UNKNOWN',
        'step3_diuretic':  'HOLD',
        'step4_raas':      'HOLD',
        'step5_bb':        'HOLD',
        'step6_sglt2':     'HOLD',
        'step6_mra':       'HOLD',
        'step7_trajectory':'UNKNOWN',
        'alert':            False,
        'alert_reason':     '',
    }

    # ── Read sensor values ──────────────────────────────────────
    hr         = get(row, 'heart_rate')
    sbp        = get(row, 'sbp')
    spo2       = get(row, 'spo2')
    resp       = get(row, 'resp_rate')
    weight     = get(row, 'weight_kg')
    creat      = get(row, 'creatinine')
    potassium  = get(row, 'potassium')
    egfr       = get(row, 'egfr')
    has_afib   = get(row, 'has_afib',  0)
    has_t1dm   = get(row, 'has_t1dm',  0)
    has_copd   = get(row, 'has_copd',  0)

    # ── STEP 1: Emergency Gates ─────────────────────────────────
    reasons = []
    if not pd.isna(spo2)      and spo2       < 90:   reasons.append(f"SpO2={spo2:.0f}%")
    if not pd.isna(sbp)       and sbp        < 90:   reasons.append(f"SBP={sbp:.0f}mmHg")
    if not pd.isna(potassium) and potassium  > 6.0:  reasons.append(f"K+={potassium:.1f}")
    if not pd.isna(creat)     and creat      > 3.5:  reasons.append(f"Creat={creat:.1f}")
    if not pd.isna(hr)        and hr         < 40:   reasons.append(f"HR={hr:.0f}bpm")
    if not pd.isna(egfr)      and egfr       < 15:   reasons.append(f"eGFR={egfr:.0f}")

    if reasons:
        result['step1_emergency'] = 'EMERGENCY'
        result['alert']           = True
        result['alert_reason']    = ' | '.join(reasons)
        # Stop here — hold everything
        for key in ['step3_diuretic','step4_raas','step5_bb','step6_sglt2','step6_mra']:
            result[key] = 'HOLD_EMERGENCY'
        return result

    # ── STEP 2: Fluid Classification ────────────────────────────
    # Proxy: use resp_rate as fluid indicator (no impedance in MIMIC)
    # resp > 22 = likely wet | resp < 16 = likely dry | middle = borderline
    if not pd.isna(resp):
        if   resp > 22: result['step2_fluid'] = 'WET'
        elif resp < 16: result['step2_fluid'] = 'DRY'
        else:           result['step2_fluid'] = 'BORDERLINE'
    fluid = result['step2_fluid']

    # ── STEP 3: Diuretic Decision ────────────────────────────────
    if fluid == 'WET':
        if not pd.isna(creat) and creat > 2.0:
            result['step3_diuretic'] = 'ESCALATE'   # wet + kidney struggling
        elif not pd.isna(potassium) and potassium < 3.5:
            result['step3_diuretic'] = 'REDUCE'     # low K+
        else:
            result['step3_diuretic'] = 'INCREASE'
    elif fluid == 'DRY':
        if not pd.isna(creat) and creat > 1.5:
            result['step3_diuretic'] = 'REDUCE'     # over-diuresis
        else:
            result['step3_diuretic'] = 'HOLD'
    else:
        result['step3_diuretic'] = 'HOLD'

    # ── STEP 4: RAAS Decision ────────────────────────────────────
    raas_gate_sbp  = pd.isna(sbp)       or sbp       >= 100
    raas_gate_k    = pd.isna(potassium) or potassium  < 5.5
    raas_gate_egfr = pd.isna(egfr)      or egfr       >= 30

    if raas_gate_sbp and raas_gate_k and raas_gate_egfr:
        result['step4_raas'] = 'UPTITRATE'
    else:
        failed = []
        if not raas_gate_sbp:  failed.append(f"SBP={sbp:.0f}<100")
        if not raas_gate_k:    failed.append(f"K+={potassium:.1f}>=5.5")
        if not raas_gate_egfr: failed.append(f"eGFR={egfr:.0f}<30")
        result['step4_raas'] = f"HOLD ({', '.join(failed)})"

    # ── STEP 5: Beta Blocker Decision ───────────────────────────
    if fluid == 'WET' or fluid == 'BORDERLINE':
        result['step5_bb'] = 'SKIP (dry before you try)'
    else:
        # DRY — check HR vs target
        hr_target = 110 if has_afib else 70
        if not pd.isna(hr):
            if hr > hr_target:
                if has_copd:
                    result['step5_bb'] = 'UPTITRATE (Bisoprolol/Metoprolol only)'
                else:
                    result['step5_bb'] = 'UPTITRATE'
            elif hr < 50:
                result['step5_bb'] = 'REDUCE'
            elif 50 <= hr <= 60:
                result['step5_bb'] = 'HOLD (HR borderline low)'
            else:
                result['step5_bb'] = 'HOLD (HR at target)'
        else:
            result['step5_bb'] = 'HOLD (no HR data)'

    # ── STEP 6: SGLT2 + MRA ─────────────────────────────────────
    # SGLT2
    if has_t1dm:
        result['step6_sglt2'] = 'CONTRAINDICATED (T1DM)'
    elif not pd.isna(egfr) and egfr < 20:
        result['step6_sglt2'] = 'HOLD (eGFR<20)'
    else:
        result['step6_sglt2'] = 'MAINTAIN/START 10mg'

    # MRA
    mra_k_ok    = pd.isna(potassium) or potassium < 5.0
    mra_egfr_ok = pd.isna(egfr)      or egfr      >= 30
    mra_cr_ok   = pd.isna(creat)     or creat      <= 2.5

    if mra_k_ok and mra_egfr_ok and mra_cr_ok:
        result['step6_mra'] = 'MAINTAIN/ADD'
    elif not pd.isna(potassium) and potassium > 5.5:
        result['step6_mra'] = 'REDUCE'
    else:
        result['step6_mra'] = 'HOLD'

    # ── STEP 7: Trajectory (simplified — flag high HR trend) ────
    if not pd.isna(hr) and not pd.isna(sbp):
        if hr > 100 and sbp < 100:
            result['step7_trajectory'] = 'WORSENING'
        elif hr < 80 and sbp > 110:
            result['step7_trajectory'] = 'IMPROVING'
        else:
            result['step7_trajectory'] = 'STABLE'

    return result

# ── Apply logic to every row ─────────────────────────────────────
print("\nRunning 7-step logic engine on 100 patients...")
results = sample.apply(run_logic, axis=1, result_type='expand')

print(f"\n{'='*55}")
print(f"✅ LOGIC ENGINE RESULTS")
print(f"{'='*55}")
print(f"  Total decisions made: {len(results):,}")
print(f"\n  Step 1 — Emergency:")
print(results['step1_emergency'].value_counts().to_string())
print(f"\n  Step 2 — Fluid Status:")
print(results['step2_fluid'].value_counts().to_string())
print(f"\n  Step 3 — Diuretic:")
print(results['step3_diuretic'].value_counts().to_string())
print(f"\n  Step 4 — RAAS:")
print(results['step4_raas'].value_counts().to_string())
print(f"\n  Step 5 — Beta Blocker:")
print(results['step5_bb'].value_counts().to_string())
print(f"\n  Step 6 — SGLT2:")
print(results['step6_sglt2'].value_counts().to_string())
print(f"\n  Step 6 — MRA:")
print(results['step6_mra'].value_counts().to_string())
print(f"\n  Step 7 — Trajectory:")
print(results['step7_trajectory'].value_counts().to_string())
print(f"\n  Total alerts triggered: {results['alert'].sum():,}")
print(f"\n  Sample alert reasons:")
print(results[results['alert']]['alert_reason'].value_counts().head(10).to_string())

# ── Save results ─────────────────────────────────────────────────
results.to_csv(r"F:\MIMIC-IV\hfref_logic_results_100patients.csv", index=False)
print(f"\n✅ Saved to hfref_logic_results_100patients.csv")

Loading final dataset...
  Loaded 1 rows, 1 patients
  Running logic on 1 patients (1 rows)

Running 7-step logic engine on 100 patients...

✅ LOGIC ENGINE RESULTS
  Total decisions made: 1

  Step 1 — Emergency:
step1_emergency
SAFE    1

  Step 2 — Fluid Status:
step2_fluid
WET    1

  Step 3 — Diuretic:
step3_diuretic
INCREASE    1

  Step 4 — RAAS:
step4_raas
UPTITRATE    1

  Step 5 — Beta Blocker:
step5_bb
SKIP (dry before you try)    1

  Step 6 — SGLT2:
step6_sglt2
MAINTAIN/START 10mg    1

  Step 6 — MRA:
step6_mra
MAINTAIN/ADD    1

  Step 7 — Trajectory:
step7_trajectory
STABLE    1

  Total alerts triggered: 0

  Sample alert reasons:
Series([], )

✅ Saved to hfref_logic_results_100patients.csv
